##### (a)

In [118]:
import collections

import numpy as np

import src.util as util
import src.svm as svm

In [119]:
train_messages, train_labels = util.load_spam_dataset('data/ds6_train.tsv')
val_messages, val_labels = util.load_spam_dataset('data/ds6_val.tsv')
test_messages, test_labels = util.load_spam_dataset('data/ds6_test.tsv')

In [120]:
def get_words(message):
    """Get the normalized list of words from a message string.

    This function should split a message into words, normalize them, and return
    the resulting list. For splitting, you should split on spaces. For normalization,
    you should convert everything to lowercase.

    Args:
        message: A string containing an SMS message

    Returns:
       The list of normalized words from the message.
    """

    # *** START CODE HERE ***
    return message.lower().split()
    # *** END CODE HERE ***


def create_dictionary(messages):
    """Create a dictionary mapping words to integer indices.

    This function should create a dictionary of word to indices using the provided
    training messages. Use get_words to process each message. 

    Rare words are often not useful for modeling. Please only add words to the dictionary
    if they occur in at least five messages.

    Args:
        messages: A list of strings containing SMS messages

    Returns:
        A python dict mapping words to integers.
    """

    # *** START CODE HERE ***
    words = [word for message in messages for word in get_words(message)]
    
    counter = collections.Counter(words)
    valid = [word for word, count in counter.items() if count >= 5]
    return {valid[i] : i for i in range(len(valid))}
    # *** END CODE HERE ***



In [121]:
dictionary = create_dictionary(train_messages)
util.write_json('./output/p06_dictionary', dictionary)

In [122]:
def transform_text(messages, word_dictionary):
    """Transform a list of text messages into a numpy array for further processing.

    This function should create a numpy array that contains the number of times each word
    appears in each message. Each row in the resulting array should correspond to each 
    message and each column should correspond to a word.

    Use the provided word dictionary to map words to column indices. Ignore words that 
    are not present in the dictionary. Use get_words to get the words for a message.

    Args:
        messages: A list of strings where each string is an SMS message.
        word_dictionary: A python dict mapping words to integers.

    Returns:
        A numpy array marking the words present in each message.
    """
    # *** START CODE HERE ***
    n, m = len(messages), len(word_dictionary)
    freqs = np.zeros((n,m), dtype=int)

    for i in range(n):
        words = get_words(messages[i])
        for word in words:
            if word in word_dictionary:
                freqs[i][word_dictionary[word]] += 1
            
    
    return freqs
    # *** END CODE HERE ***

In [123]:
train_matrix = transform_text(train_messages, dictionary)
val_matrix = transform_text(val_messages, dictionary)
test_matrix = transform_text(test_messages, dictionary)

In [124]:
def fit_naive_bayes_model(matrix, labels):
    """Fit a naive bayes model.

    This function should fit a Naive Bayes model given a training matrix and labels.

    The function should return the state of that model.

    Feel free to use whatever datatype you wish for the state of the model.

    Args:
        matrix: A numpy array containing word counts for the training data
        labels: The binary (0 or 1) labels for that training data

    Returns: The trained model
    """

    # *** START CODE HERE ***
    m, n = matrix.shape
    
    phi_y = np.mean(labels)
    phi_j_y1 = (1 + matrix[labels==1].sum(axis=0)) / (n + np.sum(matrix[labels==1]))
    phi_j_y0 = (1 + matrix[labels==0].sum(axis=0)) / (n + np.sum(matrix[labels==0]))
    
    return phi_y, phi_j_y1, phi_j_y0
    
    # *** END CODE HERE ***

In [125]:
naive_bayes_model = fit_naive_bayes_model(train_matrix, train_labels)

In [126]:
print(naive_bayes_model)

(np.float64(0.13708772717074266), array([7.77302759e-05, 6.14069180e-03, 7.77302759e-05, ...,
       4.66381656e-04, 3.88651380e-04, 7.77302759e-05], shape=(1757,)), array([2.82228301e-04, 2.84399288e-03, 1.34601190e-03, ...,
       2.17098693e-05, 4.34197386e-05, 1.51969085e-04], shape=(1757,)))


In [127]:
def predict_from_naive_bayes_model(model, matrix):
    """Use a Naive Bayes model to compute predictions for a target matrix.

    This function should be able to predict on the models that fit_naive_bayes_model
    outputs.

    Args:
        model: A trained model from fit_naive_bayes_model
        matrix: A numpy array containing word counts

    Returns: A numpy array containg the predictions from the model
    """
    # *** START CODE HERE ***
    
    phi_y, phi_j_y1, phi_j_y0 = model
    
    return (matrix @ (np.log(phi_j_y1)-np.log(phi_j_y0)) + np.log(phi_y) - np.log(1-phi_y)) >= 0
    # *** END CODE HERE ***

In [128]:
naive_bayes_predictions = predict_from_naive_bayes_model(naive_bayes_model, test_matrix)

np.savetxt('./output/p06_naive_bayes_predictions', naive_bayes_predictions)

naive_bayes_accuracy = np.mean(naive_bayes_predictions == test_labels)

print('Naive Bayes had an accuracy of {} on the testing set'.format(naive_bayes_accuracy))

Naive Bayes had an accuracy of 0.978494623655914 on the testing set


In [129]:
def get_top_five_naive_bayes_words(model, dictionary):
    """Compute the top five words that are most indicative of the spam (i.e positive) class.

    Ues the metric given in 6c as a measure of how indicative a word is.
    Return the words in sorted form, with the most indicative word first.

    Args:
        model: The Naive Bayes model returned from fit_naive_bayes_model
        dictionary: A mapping of word to integer ids

    Returns: The top five most indicative words in sorted order with the most indicative first
    """
    # *** START CODE HERE ***
    
    phi_y, phi_j_y1, phi_j_y0 = model
    
    newd = {v : k for k,v in dictionary.items()}
    
    metric = np.argsort(np.log(phi_j_y1)-np.log(phi_j_y0))[-5:]
    
    return [newd[i] for i in metric]
    
    # *** END CODE HERE ***

In [130]:
top_5_words = get_top_five_naive_bayes_words(naive_bayes_model, dictionary)

print('The top 5 indicative words for Naive Bayes are: ', top_5_words)

util.write_json('./output/p06_top_indicative_words', top_5_words)

The top 5 indicative words for Naive Bayes are:  ['urgent!', 'tone', 'prize', 'won', 'claim']


In [134]:
def compute_best_svm_radius(train_matrix, train_labels, val_matrix, val_labels, radius_to_consider):
    """Compute the optimal SVM radius using the provided training and evaluation datasets.

    You should only consider radius values within the radius_to_consider list.
    You should use accuracy as a metric for comparing the different radius values.

    Args:
        train_matrix: The word counts for the training data
        train_labels: The spma or not spam labels for the training data
        val_matrix: The word counts for the validation data
        val_labels: The spam or not spam labels for the validation data
        radius_to_consider: The radius values to consider
    
    Returns:
        The best radius which maximizes SVM accuracy.
    """
    # *** START CODE HERE ***
    r_best = (radius_to_consider[0], 0)
    
    for r in radius_to_consider:
        output = svm.train_and_predict_svm(train_matrix, train_labels, val_matrix, r)
        score = np.sum(val_labels==output)
        
        if score > r_best[1]:
            r_best = (r, score)
    
    return r_best[0]
    
    
    
    # *** END CODE HERE ***

In [135]:
optimal_radius = compute_best_svm_radius(train_matrix, train_labels, val_matrix, val_labels, [0.01, 0.1, 1, 10])

util.write_json('./output/p06_optimal_radius', optimal_radius)

print('The optimal SVM radius was {}'.format(optimal_radius))

svm_predictions = svm.train_and_predict_svm(train_matrix, train_labels, test_matrix, optimal_radius)

svm_accuracy = np.mean(svm_predictions == test_labels)

print('The SVM model had an accuracy of {} on the testing set'.format(svm_accuracy, optimal_radius))

The optimal SVM radius was 0.1
The SVM model had an accuracy of 0.967741935483871 on the testing set


In [133]:
train_matrix, train_labels

(array([[1, 1, 1, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 1, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(4457, 1757)),
 array([0, 0, 0, ..., 0, 0, 0], shape=(4457,)))